# CASE1: what goes wrong in Antoine's method, and why

Everything here is CASE1 (cutoff 2002, designed switch in 2005: cancer before, infectious disease after),
monitoring one triple type, **Gene - causes_or_contributes_to_condition - Disease**, exactly as in Antoine's
notebook. The knowledge graph is built from every triple up to 2002 (330,740 triples).

| Section | Claim | Data |
|---|---|---|
| 1 | Antoine's sphere rule never retrains the OCSVM | the 20 Phase 1 runs (`runs/`) |
| 2 | After the first one, every alarm happens exactly at a re-embed, identical for every model and for Antoine's own run | the 20 Phase 1 runs |
| 3 | The refill after an alarm skips most of the stream | the 20 Phase 1 runs |
| 4 | **Experiment A**: re-embedding the *same* graph gives a new map; the old OCSVM calls the same triples outliers, while the sphere rule sees nothing | new: 2 TransE trainings |
| 5 | **Experiment B**: with no change at all, the test alarms, because the reference is judged on its own training data | new: reuses Experiment A |
| 6 | **Experiment C**: training length (2 epochs as Antoine vs the chosen epochs) does not change the sphere-rule alarms, but does change the always-retrain results | new: 20 short runs |

Phase 1 settings: 64 dimensions, StandardScaler, OCSVM nu 0.1 gamma 'scale', window 250, alpha 0.025, keep 10% after an alarm.
Embeddings trained with the learning rate and epochs chosen by early stopping on MRR (`configs.TRAINING`).

**Running it:** Run All. Sections 1-3 take seconds. Experiments A and B train TransE twice on the CPU
(roughly 10-60 minutes in total); Experiment C runs 20 short stream runs (roughly 1-2 hours).
Slow results are saved in `notebook_cache/` and `runs_2epochs/`, so running the notebook again is fast.

In [ ]:
import json, os, pickle, subprocess, sys, time, warnings, logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
logging.getLogger('pykeen').setLevel(logging.ERROR)

HERE = Path.cwd()
if not (HERE / 'drift_sphere.py').exists():
    HERE = HERE / 'phase1'
sys.path.insert(0, str(HERE))
import configs
import drift_sphere as ds

DATA_DIR = Path(os.environ.get('NB_DATA_DIR', HERE.parent / 'data'))
RUNS = Path(os.environ.get('NB_RUNS', HERE / 'runs'))
RUNS_2EP = Path(os.environ.get('NB_RUNS_2EP', HERE / 'runs_2epochs'))
CACHE = Path(os.environ.get('NB_CACHE', HERE / 'notebook_cache'))
CACHE.mkdir(parents=True, exist_ok=True)
ADAZOR = HERE.parent / 'adazor' / 'src'
TEST = os.environ.get('NB_TEST') == '1'          # tiny smoke test only

P = configs.PRESETS['phase1']
TRAIN_TRANSE = configs.TRAINING[('case1', 'TransE')]
SWITCH_YEAR = 2005
ANTOINE_ROWS = [2295, 2857, 3430, 37769]   # file rows of the 4 alarms in Antoine's own concatenation run
SWITCH_INDEX = 567                          # first 2005 triple of the CASE1 Gene-Disease stream (566 come before it)
try:
    import torch
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
except ImportError:
    DEVICE = 'cpu'

CUR, REF, INK, INK2, GRID, SURF = '#2a78d6', '#eb6834', '#0b0b0b', '#52514e', '#e4e3df', '#fcfcfb'
plt.rcParams.update({'figure.facecolor': SURF, 'axes.facecolor': SURF, 'axes.edgecolor': INK2,
                     'axes.labelcolor': INK2, 'xtick.color': INK2, 'ytick.color': INK2, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
print('device:', DEVICE, '| data:', DATA_DIR, '| runs:', RUNS)

In [ ]:
def read_tsv(path):
    try:
        return pd.read_csv(path, sep='\t')
    except (FileNotFoundError, pd.errors.EmptyDataError):   # a run with no alarms writes an empty file
        return pd.DataFrame(columns=['stream_index', 'row_number', 'event_year', 'trigger_reason',
                                     'center_radius_ratio', 'reference_outlier_fraction', 'current_outlier_fraction',
                                     'reembedded', 'drift_detected'])


def load_runs(folder, dataset='case1'):
    out = {}
    for d in sorted(folder.glob(f'{dataset}__*__phase1')):
        if '__sphereR' in d.name or not (d / 'summary.json').exists():
            continue
        s = json.loads((d / 'summary.json').read_text())
        out[(s['model'], s['composition'], s['retrain_policy'])] = {
            'summary': s, 'drifts': read_tsv(d / 'drifts.tsv'),
            'history': read_tsv(d / 'window_history.tsv.gz'),
            'shifts': read_tsv(d / 'embedding_shift_metrics.tsv')}
    return out


R = load_runs(RUNS)
print(len(R), 'runs loaded:', sorted({k[2] for k in R}))

## 1. Antoine's sphere rule never retrains the OCSVM

When a triple brings a new entity, the graph is re-embedded from scratch. The rule then computes
**ratio = distance of the new triple from the centre of the box around all graph triples / half the box diagonal**,
and retrains the OCSVM only if ratio > k (Antoine: k = 1). Any point inside the box has ratio <= 1, and the box
is drawn around all 330,740 graph triples, so a new triple essentially never lands outside it.
The ratio is logged at every new-entity re-embed in both the sphere runs and the always-retrain runs.

In [ ]:
rows, all_ratios = [], []
for (m, c, rule), r in R.items():
    sh = r['shifts']
    ratios = sh.loc[sh['trigger_reason'].eq('unseen_element'), 'center_radius_ratio'].dropna() if len(sh) else pd.Series(dtype=float)
    all_ratios += [(m, c, rule, x) for x in ratios]
    rows.append({'model': m, 'composition': c, 'rule': rule,
                 'new-entity re-embeds': r['summary']['num_unseen_refresh_events'],
                 'OCSVM retrained by the rule': r['summary']['num_threshold_retrains'] if rule == 'sphere' else 'always',
                 'largest ratio': round(ratios.max(), 3) if len(ratios) else None,
                 'ratios > 1': int((ratios > 1).sum())})
t1 = pd.DataFrame(rows).sort_values(['rule', 'model', 'composition']).reset_index(drop=True)
display(t1) if 'display' in dir() else print(t1.to_string())
ratios_all = pd.DataFrame(all_ratios, columns=['model', 'composition', 'rule', 'ratio'])
print(f"\n{len(ratios_all)} logged ratios in total; largest {ratios_all.ratio.max():.3f}; above 1: {(ratios_all.ratio > 1).sum()}")
print('Sphere rule (k = 1): OCSVM retrains in all sphere runs =', int(t1.loc[t1.rule.eq('sphere'), 'OCSVM retrained by the rule'].sum()))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
models = configs.MODELS
for i, m in enumerate(models):
    v = ratios_all.loc[ratios_all.model.eq(m), 'ratio'].to_numpy()
    jitter = np.random.default_rng(i).uniform(-0.25, 0.25, len(v))
    ax.scatter(v, np.full(len(v), i) + jitter, s=14, color=CUR, alpha=0.6, edgecolor='none')
ax.axvline(1.0, color=INK, lw=1.5, ls=(0, (4, 3)))
ax.text(0.99, len(models) - 0.4, 'k = 1: retrain only right of this line', ha='right', fontsize=9, color=INK2)
ax.set_yticks(range(len(models)), models); ax.set_xlim(0, 1.1)
ax.set_xlabel('sphere ratio of each new triple (all re-embeds, both compositions, both rules)')
ax.set_title('Every ratio is far below 1, so the rule never fires', loc='left', fontsize=11)
ax.grid(axis='x', color=GRID); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

## 2. The alarms come from re-embedding, not from drift

Below: every alarm of every run, with the file row of the triple, its year, whether the graph was re-embedded
at that step, and the outlier rates (current window vs reference). An alarm fires when the two differ by more
than about 0.10. The last column of the second table checks whether a run alarms at **exactly the same 4 file rows
as Antoine's own run** (2295, 2857, 3430, 37769; his notebook: 2 epochs, 32 dimensions, no scaling).

In [ ]:
alarm_rows = []
for (m, c, rule), r in R.items():
    h = r['history'].set_index('stream_index')
    for a in r['drifts'].itertuples():
        alarm_rows.append({'model': m, 'composition': c, 'rule': rule, 'stream triple': int(a.stream_index),
                           'file row': int(a.row_number), 'year': int(a.event_year),
                           're-embedded at this step': bool(h.loc[a.stream_index, 'reembedded']),
                           'reference rate': round(a.reference_outlier_fraction, 2),
                           'current rate': round(a.current_outlier_fraction, 2)})
alarms = pd.DataFrame(alarm_rows).sort_values(['rule', 'model', 'composition', 'stream triple'])
show = alarms[alarms.rule.eq('sphere')]
display(show) if 'display' in dir() else print(show.to_string(index=False))

In [ ]:
per_run = alarms.groupby(['rule', 'model', 'composition']).agg(
    alarms=('file row', 'size'),
    before_2005=('year', lambda y: int((y < SWITCH_YEAR).sum())),
    file_rows=('file row', lambda v: list(v)),
    alarms_at_a_reembed=('re-embedded at this step', 'sum')).reset_index()
per_run['same rows as Antoine'] = per_run['file_rows'].apply(lambda v: v == ANTOINE_ROWS)
display(per_run) if 'display' in dir() else print(per_run.to_string(index=False))
sp = per_run[per_run.rule.eq('sphere')]
print(f"\nSphere runs with exactly Antoine's 4 alarm rows: {int(sp['same rows as Antoine'].sum())} of {len(sp)}")
print(f"Sphere-rule alarms that happen at a re-embed step: {int(sp.alarms_at_a_reembed.sum())} of {int(sp.alarms.sum())} "
      f"(the remaining ones are the very first window)")

In [ ]:
def plot_run(model='TransE', comp='hadamard'):
    fig, axes = plt.subplots(2, 1, figsize=(10, 6.4), sharex=True)
    for ax, (rule, title) in zip(axes, [('sphere', "Antoine's sphere rule (k = 1): the OCSVM is never retrained"),
                                         ('always', 'Always retrain the OCSVM when the graph is re-embedded')]):
        r = R[(model, comp, rule)]; w = r['history']; n = r['summary']['future_rows']
        idx = w['stream_index'].to_numpy()
        tested = np.zeros(n + 2, bool); tested[idx] = True
        start = None
        for i in range(1, n + 1):
            if not tested[i] and start is None:
                start = i
            if (tested[i] or i == n) and start is not None:
                ax.axvspan(start - 0.5, i - 0.5, color='#efeeea', lw=0, zorder=0); start = None
        for k, seg in enumerate(np.split(np.arange(len(w)), np.where(np.diff(idx) > 1)[0] + 1)):
            s = w.iloc[seg]
            ax.plot(s.stream_index, s.current_outlier_fraction, color=CUR, lw=2, label='outlier rate, current window' if k == 0 else None)
            ax.plot(s.stream_index, s.reference_outlier_fraction, color=REF, lw=2, label='outlier rate, reference' if k == 0 else None)
        re = w[w.reembedded]
        ax.plot(re.stream_index, np.full(len(re), -0.06), '|', color=INK2, ms=9, mew=1.5, clip_on=False, label='graph re-embedded')
        al = w[w.drift_detected]
        ax.scatter(al.stream_index, al.current_outlier_fraction, s=70, color=INK, edgecolor=SURF, lw=2, zorder=5, label='alarm')
        for a in al.itertuples():
            ax.annotate(str(int(a.event_year)), (a.stream_index, a.current_outlier_fraction), xytext=(6, 4), textcoords='offset points', fontsize=8.5, color=INK2)
        ax.axvline(SWITCH_INDEX, color=INK2, ls=(0, (4, 3)), lw=1.2)
        ax.text(SWITCH_INDEX + 6, 1.03, 'designed drift (2005)', fontsize=8.5, color=INK2, va='bottom')
        ax.text(0.012, 0.9, f"{len(al)} alarms, {int((al.event_year < SWITCH_YEAR).sum())} before 2005\n{len(w)} of {n} triples tested",
                transform=ax.transAxes, va='top', fontsize=8.5, color=INK2, linespacing=1.5)
        ax.set_title(title, loc='left', fontsize=11); ax.set_ylim(-0.1, 1.08); ax.set_xlim(0, n + 5)
        ax.set_ylabel('share of outliers'); ax.grid(axis='y', color=GRID); ax.set_axisbelow(True)
    axes[1].set_xlabel('Gene-Disease stream triple (2003 onward); grey = skipped by the refill after an alarm')
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc='upper center', ncol=4, frameon=False, fontsize=9, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(f'CASE1, {model} {comp}', x=0.01, ha='left', y=1.07, fontsize=12)
    plt.tight_layout(); plt.show()


plot_run('TransE', 'hadamard')

Reading the figure: under the sphere rule, every later alarm sits exactly on a re-embed tick, where the current
outlier rate jumps from about 0.17 to about 1.00 in a single step: the graph got a new map and the OCSVM, trained on the
old map, no longer recognises anything. With always-retrain those jumps disappear. In both panels the first alarm
(2003) happens before any re-embed: that is Section 5.

## 3. The refill after an alarm skips most of the stream

After an alarm, Antoine's code keeps 10% of the window (25 triples) and refills it with the next 225 stream triples,
which are added to the graph but never tested. On a 763-triple stream, 3-4 alarms use the stream up.

In [ ]:
rows = []
for (m, c, rule), r in R.items():
    w = r['history']
    rows.append({'model': m, 'composition': c, 'rule': rule, 'stream triples': r['summary']['future_rows'],
                 'tested': len(w), 'tested from 2005 on': int((w.event_year >= SWITCH_YEAR).sum()),
                 'tested in 2005-2006': int(w.event_year.between(2005, 2006).sum())})
t3 = pd.DataFrame(rows).sort_values(['rule', 'model', 'composition']).reset_index(drop=True)
display(t3) if 'display' in dir() else print(t3.to_string(index=False))

## 4. Experiment A: re-embedding the same graph gives a new map

Nothing changes in the data. TransE is trained **twice on exactly the same CASE1 graph** (all triples up to 2002),
with the Phase 1 settings; only the random seed differs (42, 43). Then:

* the OCSVM is trained on the reference triples (the first 256 Gene-Disease triples before the cutoff) under embedding 1,
  exactly as in Phase 1;
* it scores **the same triples** and the 250-triple window under embedding 2;
* the sphere rule's ratio is computed for the window triples under embedding 2 against the box under embedding 1,
  which is exactly what the rule does at a re-embed;
* finally embedding 2 is rotated onto embedding 1 (orthogonal Procrustes, fitted on 20,000 graph triples)
  and the window is scored again.

If re-embedding were harmless, the outlier rates under embedding 2 would match those under embedding 1.

In [ ]:
dcfg = configs.DATASETS['case1']
triples_df = ds.load_temporal_triples(DATA_DIR / dcfg['file'])
initial_kg, initial_stream, future_stream, lookup = ds.build_initial_knowledge_graph(
    triples_df, dcfg['cutoff_year'], dcfg['target_type'])
KG = ds.KnowledgeGraph(initial_kg)
init_rows = initial_stream.to_dict('records')
N_WIN = min(P['window_size'], len(init_rows) // 2)   # 250 on the real data (506 triples), as in Phase 1
REF_ROWS, WIN_ROWS = init_rows[:-N_WIN], init_rows[-N_WIN:]
print(f'graph {len(KG)} triples | Gene-Disease before the cutoff {len(init_rows)} '
      f'= reference {len(REF_ROWS)} + window {len(WIN_ROWS)}')


def embed(model='TransE', seed=42, epochs=None, lr=None):
    tr = configs.TRAINING[('case1', model)]
    epochs = epochs or (3 if TEST else tr['epochs']); lr = lr or tr['learning_rate']
    f = CACHE / f"case1_{model}_seed{seed}_e{epochs}_lr{lr:g}{'_TEST' if TEST else ''}.pkl"
    if f.exists():
        return pickle.loads(f.read_bytes())
    cfg = {'embedder': 'pykeen', 'model': model, 'real_dim': P['real_dim'], 'epochs': epochs,
           'learning_rate': lr, 'batch_size': P['batch_size'], 'seed': seed, 'device': DEVICE}
    t0 = time.time(); print(f'training {model} seed {seed}, {epochs} epochs ...', flush=True)
    st = ds.generate_embeddings_from_kg(KG, cfg)
    print(f'  done in {(time.time() - t0) / 60:.1f} min', flush=True)
    f.write_bytes(pickle.dumps(st)); return st


E1 = embed(seed=42)
E2 = embed(seed=43)
shared = [e for e in E1.entity_index if e in E2.entity_index]
a = E1.entity_matrix[[E1.entity_index[e] for e in shared]]; b = E2.entity_matrix[[E2.entity_index[e] for e in shared]]
print(f'same entity, two trainings: mean cosine similarity {np.mean(np.sum(a * b, axis=1)):.3f} '
      f'(1 = same vector, 0 = unrelated)')

In [ ]:
def procrustes_map(X_from, X_to):
    mf, mt = X_from.mean(0), X_to.mean(0)
    U, _, Vt = np.linalg.svd((X_from - mf).T @ (X_to - mt))
    Rm = U @ Vt
    return lambda X: (X - mf) @ Rm + mt


def detector_for(flags):
    det, _ = ds.make_detector('ztest', P['alpha'], ADAZOR)
    det.reset(flags); return det


rng = np.random.default_rng(0)
all_kg = KG.triples()
sample = all_kg[rng.choice(len(all_kg), min(20000, len(all_kg)), replace=False)]
rowsA = []
for comp in configs.COMPOSITIONS:
    f = lambda rows, E: ds.rows_to_features(rows, E, comp)
    ocsvm, scaler, ref_flags = ds.train_classifier(f(REF_ROWS, E1), P)
    det = detector_for(ref_flags)
    rate = lambda X: float(ds.outlier_flags(ocsvm, scaler, X).mean())
    alarm = lambda X: bool(det.detect(ds.outlier_flags(ocsvm, scaler, X)))
    align = procrustes_map(ds.compose(sample[:, 0], sample[:, 1], sample[:, 2], E2, comp),
                           ds.compose(sample[:, 0], sample[:, 1], sample[:, 2], E1, comp))
    box1 = ds.compute_feature_space_boundary_from_kg(KG, E1, comp)
    win2 = f(WIN_ROWS, E2)
    ratios = [ds.compute_center_radius_metrics(v, box1, 1.0)['center_radius_ratio'] for v in win2]
    for label, X in [('reference, embedding 1 (its own training data)', f(REF_ROWS, E1)),
                     ('window, embedding 1 (same map)', f(WIN_ROWS, E1)),
                     ('reference, embedding 2 (same triples, new map)', f(REF_ROWS, E2)),
                     ('window, embedding 2 (new map)', win2),
                     ('window, embedding 2 rotated onto embedding 1', align(win2))]:
        rowsA.append({'composition': comp, 'triples scored': label, 'outlier rate': round(rate(X), 3),
                      "Antoine's test alarms": '' if 'own training' in label else alarm(X)})
    rowsA.append({'composition': comp, 'triples scored': 'sphere rule on the window under embedding 2',
                  'outlier rate': f'largest ratio {max(ratios):.2f} (> 1 needed)',
                  "Antoine's test alarms": 'retrain OCSVM? ' + ('yes' if max(ratios) > 1 else 'no')})
tA = pd.DataFrame(rowsA)
display(tA) if 'display' in dir() else print(tA.to_string(index=False))

What to look for: under embedding 2 the OCSVM calls a large share of **the very same triples** outliers and the
test alarms, although nothing in the data changed. At the same time the sphere rule's ratio stays below 1, so the
rule says "no need to retrain". The two disagree completely: the box check cannot see a change of map, the OCSVM
sees nothing else. Rotating embedding 2 onto embedding 1 brings the rates back towards embedding 1's,
which shows the problem is the coordinate system, not the data.

## 5. Experiment B: it alarms when nothing changes

One embedding (embedding 1 from Experiment A), no re-embedding, and only the 506 Gene-Disease triples from
**before the cutoff**, so there is no drift and no new entity. 100 times: shuffle them, use 256 as the reference and
250 as the window, train the OCSVM, and run Antoine's test.

* **Antoine's reference:** the detector's reference rate is the OCSVM's outlier rate on its own training triples.
* **Fair reference:** the 256 reference triples are split into 128 to train the OCSVM and 128 that it has not seen;
  the detector's reference rate is measured on those 128.

In [ ]:
N_REP = 20 if TEST else 100
rowsB = []
for comp in configs.COMPOSITIONS:
    X_all = ds.rows_to_features(init_rows, E1, comp)
    n_ref = len(REF_ROWS)
    for rep in range(N_REP):
        perm = np.random.default_rng(rep).permutation(len(X_all))
        Xr, Xw = X_all[perm[:n_ref]], X_all[perm[n_ref:]]
        oc, sc, own = ds.train_classifier(Xr, P)
        win_flags = ds.outlier_flags(oc, sc, Xw)
        half = n_ref // 2
        oc2, sc2, _ = ds.train_classifier(Xr[:half], P)
        cal_flags = ds.outlier_flags(oc2, sc2, Xr[half:])
        rowsB.append({'composition': comp, 'rep': rep, 'own training rate': own.mean(), 'window rate': win_flags.mean(),
                      'alarm (Antoine)': bool(detector_for(own).detect(win_flags)),
                      'unseen-reference rate': cal_flags.mean(),
                      'alarm (fair reference)': bool(detector_for(cal_flags).detect(ds.outlier_flags(oc2, sc2, Xw)))})
B = pd.DataFrame(rowsB)
tB = B.groupby('composition').agg(**{
    'repeats': ('rep', 'size'),
    'outlier rate on own training triples': ('own training rate', 'mean'),
    'outlier rate on unseen triples (window)': ('window rate', 'mean'),
    "alarms, Antoine's reference (%)": ('alarm (Antoine)', lambda x: 100 * x.mean()),
    'outlier rate on unseen reference': ('unseen-reference rate', 'mean'),
    'alarms, fair reference (%)': ('alarm (fair reference)', lambda x: 100 * x.mean())}).round(3)
display(tB) if 'display' in dir() else print(tB.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, comp in zip(axes, configs.COMPOSITIONS):
    d = B[B.composition.eq(comp)]
    bins = np.linspace(0, 1, 41)
    ax.hist(d['own training rate'], bins=bins, color=REF, alpha=0.85, label='own training triples')
    ax.hist(d['window rate'], bins=bins, color=CUR, alpha=0.85, label='unseen triples, same period')
    ax.set_title(comp, loc='left', fontsize=10); ax.set_xlabel('outlier rate'); ax.grid(axis='y', color=GRID)
axes[0].set_ylabel('repeats'); axes[0].legend(frameon=False, fontsize=8.5)
fig.suptitle('No drift, no re-embedding: the OCSVM still sees unseen triples as more unusual than its own', x=0.01, ha='left', fontsize=11)
plt.tight_layout(); plt.show()

## 6. Experiment C: does training length matter?

Antoine trained TransE for 2 epochs at PyKEEN's default learning rate (0.001) and never measured the result.
Here every model is trained exactly like that (2 epochs, learning rate 0.001), everything else is the Phase 1 setting,
and the CASE1 runs are repeated for both rules (20 runs, saved in `runs_2epochs/`; finished runs are skipped).

First, how much does each model learn in 2 epochs? (MRR on held-out triples; random guessing is about 0.00016.)

In [ ]:
def mrr_after(model, epochs, lr):
    f = CACHE / f"mrr_case1_{model}_e{epochs}_lr{lr:g}{'_TEST' if TEST else ''}.json"
    if f.exists():
        return json.loads(f.read_text())
    import pykeen.models as pm
    from pykeen.evaluation import RankBasedEvaluator
    from pykeen.training import SLCWATrainingLoop
    from pykeen.triples import TriplesFactory
    tf = TriplesFactory.from_labeled_triples(KG.triples().astype(str))
    train, valid, test = tf.split([0.96, 0.02, 0.02], random_state=P['seed'])
    test_m = test.mapped_triples
    if len(test_m) > 2000:
        test_m = test_m[np.random.default_rng(P['seed']).choice(len(test_m), 2000, replace=False)]
    dim = P['real_dim'] // 2 if model in configs.COMPLEX_MODELS else P['real_dim']
    m = getattr(pm, model)(triples_factory=train, embedding_dim=dim, random_seed=P['seed'])
    SLCWATrainingLoop(model=m, triples_factory=train, optimizer='Adam', optimizer_kwargs={'lr': lr}).train(
        triples_factory=train, num_epochs=epochs, batch_size=P['batch_size'], use_tqdm=False, use_tqdm_batch=False)
    res = RankBasedEvaluator(filtered=True).evaluate(m, test_m, use_tqdm=False, additional_filter_triples=[
        train.mapped_triples, valid.mapped_triples, test.mapped_triples])
    out = {'mrr': float(res.get_metric('both.realistic.inverse_harmonic_mean_rank')),
           'random': float(np.log(tf.num_entities) / tf.num_entities)}
    f.write_text(json.dumps(out)); return out


rows = []
for m in configs.MODELS:
    two = mrr_after(m, 1 if TEST else 2, 0.001)
    q = HERE / 'quality' / f'case1__{m}__phase1.json'
    chosen = json.loads(q.read_text()) if q.exists() else {}
    rows.append({'model': m, 'MRR after 2 epochs (Antoine)': round(two['mrr'], 4),
                 'x random': round(two['mrr'] / two['random']),
                 'MRR with the chosen training': round(chosen.get('mrr', np.nan), 4),
                 'chosen epochs / learning rate': f"{configs.TRAINING[('case1', m)]['epochs']} / {configs.TRAINING[('case1', m)]['learning_rate']}"})
tC1 = pd.DataFrame(rows)
display(tC1) if 'display' in dir() else print(tC1.to_string(index=False))

In [ ]:
RUN_EXPERIMENT_C = True   # set False to only read finished runs
if RUN_EXPERIMENT_C:
    cmd = [sys.executable, str(HERE / 'run_grid.py'), '--grid', 'sphere', 'always', '--datasets', 'case1',
           '--parallel', '3', '--out-dir', str(RUNS_2EP), '--data-dir', str(DATA_DIR), '--adazor-src', str(ADAZOR),
           '--', '--epochs', '1' if TEST else '2', '--learning-rate', '0.001', '--device', DEVICE]
    if TEST:
        cmd += ['--max-stream', '120']
    t0 = time.time()
    proc = subprocess.Popen(cmd, cwd=HERE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError('some runs failed: see runs_2epochs/_logs/')
    print(f'{(time.time() - t0) / 60:.1f} min')
R2 = load_runs(RUNS_2EP)
print(len(R2), 'runs with 2 epochs')

In [ ]:
def describe(runs, label):
    rows = []
    for (m, c, rule), r in runs.items():
        dr = r['drifts']; after = dr.loc[dr.event_year >= SWITCH_YEAR, 'event_year'] if len(dr) else pd.Series(dtype=float)
        rows.append({'model': m, 'composition': c, 'rule': rule, f'alarms ({label})': len(dr),
                     f'before 2005 ({label})': int((dr.event_year < SWITCH_YEAR).sum()) if len(dr) else 0,
                     f'first alarm from 2005 ({label})': int(after.min()) if len(after) else None,
                     f'alarm rows ({label})': list(dr.row_number) if len(dr) else []})
    return pd.DataFrame(rows)


cmp = describe(R2, '2 epochs').merge(describe(R, 'chosen'), on=['model', 'composition', 'rule'])
cmp['same alarm rows'] = [a == b for a, b in zip(cmp['alarm rows (2 epochs)'], cmp['alarm rows (chosen)'])]
cmp = cmp.sort_values(['rule', 'model', 'composition']).reset_index(drop=True)
cols = ['rule', 'model', 'composition', 'alarms (2 epochs)', 'alarms (chosen)', 'before 2005 (2 epochs)', 'before 2005 (chosen)',
        'first alarm from 2005 (2 epochs)', 'first alarm from 2005 (chosen)', 'same alarm rows']
display(cmp[cols]) if 'display' in dir() else print(cmp[cols].to_string(index=False))
for rule in ['sphere', 'always']:
    d = cmp[cmp.rule.eq(rule)]
    print(f"{rule}: identical alarm rows with 2 epochs and with the chosen training in {int(d['same alarm rows'].sum())} of {len(d)} runs")

What to look for: under the sphere rule the alarms should be identical whatever the training, because the rule never
lets the OCSVM learn the embedding; the result is set by re-embedding and the refill. Under always-retrain the OCSVM
works on the embedding, so a difference between the 2-epoch and the trained runs shows that embedding quality matters
once the retraining problem is removed.

## Summary (numbers from this notebook)

In [ ]:
sp = per_run[per_run.rule.eq('sphere')]
print(f"1. Sphere rule: OCSVM retrained {int(t1.loc[t1.rule.eq('sphere'), 'OCSVM retrained by the rule'].sum())} times in "
      f"{len(sp)} runs; largest of {len(ratios_all)} ratios = {ratios_all.ratio.max():.2f} (needs > 1).")
print(f"2. Sphere-rule alarms at a re-embed: {int(sp.alarms_at_a_reembed.sum())} of {int(sp.alarms.sum())}; "
      f"runs with exactly Antoine's 4 alarm rows: {int(sp['same rows as Antoine'].sum())} of {len(sp)}.")
print(f"3. Triples tested, sphere runs: {t3.loc[t3.rule.eq('sphere'), 'tested'].min()}-{t3.loc[t3.rule.eq('sphere'), 'tested'].max()} "
      f"of {t3['stream triples'].iloc[0]}; tested in 2005-2006: {t3.loc[t3.rule.eq('sphere'), 'tested in 2005-2006'].max()}.")
for comp in configs.COMPOSITIONS:
    d = tA[tA.composition.eq(comp)].set_index('triples scored')['outlier rate']
    print(f"4. ({comp}) same triples: outlier rate {d.iloc[0]} on own training data, {d.iloc[1]} unseen same map, "
          f"{d.iloc[2]} under the new map, {d.iloc[4]} after rotating back; {d.iloc[5]}.")
for comp, r in tB.iterrows():
    print(f"5. ({comp}) no change at all: Antoine's test alarms in {r.iloc[3]:.0f}% of {int(r.iloc[0])} repeats; "
          f"with a fair reference in {r.iloc[5]:.0f}%.")
for rule in ['sphere', 'always']:
    d = cmp[cmp.rule.eq(rule)]
    print(f"6. ({rule}) 2 epochs vs chosen training: identical alarm rows in {int(d['same alarm rows'].sum())} of {len(d)} runs.")